# 第十三章：极端行情模拟 — 杠杆的生死线 / Chapter 13: Crash Simulation — The Line Between Life and Liquidation

## 你将学到 / What You Will Learn
- 不同杠杆在暴跌中的命运差距有多大 / How massively different leverage levels fare in a crash
- 为什么高杠杆"看起来赚更多"但"死得也更快" / Why high leverage *looks* more profitable yet dies fastest
- 怎样的杠杆水平才算"安全" / What a "safe" level of leverage actually looks like

---

## 13.1 一个真实的问题 / 13.1 A Real-World Question

> 你在 BTC $60,000 时做多，分别用 5倍、20倍、100倍杠杆。
> BTC 暴跌 50% 到 $30,000。
>
> You go long BTC at $60,000, using 5×, 20×, and 100× leverage respectively.
> BTC then crashes 50 %, to $30,000.
>
> **问：三个账户分别还剩多少钱？**
>
> **Question: how much money is left in each account?**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 13.2 互动模拟 / 13.2 Interactive Simulation

In [ ]:
def crash_simulator(entry=60000, drop_pct=50, lev_low=5, lev_mid=20, lev_high=100):
    plt.close('all')
    target = entry * (1 - drop_pct / 100)
    prices = np.linspace(entry, target, 200)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Top-left: price path + liquidation lines
    ax = axes[0][0]
    hours = np.linspace(0, 48, len(prices))
    ax.plot(hours, prices, 'k-', lw=3)
    ax.fill_between(hours, prices, entry, color=C_RED, alpha=0.15)
    for lev, col in [(lev_low, C_GREEN), (lev_mid, C_ORANGE), (lev_high, C_RED)]:
        liq = entry * (1 - 1 / lev + 0.005)
        ax.axhline(y=liq, color=col, ls='--', lw=2, label=f'{lev}x liq ${liq:,.0f}')
    ax.set_xlabel('Hours', fontsize=11)
    ax.set_ylabel('BTC Price', fontsize=11)
    ax.set_title(f'BTC ${entry:,} drops {drop_pct}% to ${target:,.0f}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Top-right: ROI
    ax = axes[0][1]
    for lev, col in [(lev_low, C_GREEN), (lev_mid, C_ORANGE), (lev_high, C_RED)]:
        margin = entry / lev
        roi = (prices - entry) / margin * 100
        ax.plot(prices, roi, color=col, lw=2, label=f'{lev}x')
    ax.axhline(y=-100, color=C_RED, ls=':', lw=2, label='-100% (liquidation)')
    ax.axhline(y=0, color='gray', ls='--')
    ax.set_xlabel('BTC Price', fontsize=11)
    ax.set_ylabel('ROI (%)', fontsize=11)
    ax.set_title('ROI by Leverage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-150, 20)
    ax.grid(True, alpha=0.2)

    # Bottom-left: balance
    ax = axes[1][0]
    for lev, col in [(lev_low, C_GREEN), (lev_mid, C_ORANGE), (lev_high, C_RED)]:
        margin = entry / lev
        balance = np.maximum(margin + (prices - entry), 0)
        ax.plot(prices, balance, color=col, lw=2, label=f'{lev}x (capital ${margin:,.0f})')
    ax.axhline(y=0, color=C_RED, ls=':', lw=2)
    ax.set_xlabel('BTC Price', fontsize=11)
    ax.set_ylabel('Account Balance (USD)', fontsize=11)
    ax.set_title('Account Balance vs Price', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Bottom-right: result panel
    ax = axes[1][1]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Final Result', fontsize=15, fontweight='bold')

    for i, (lev, col) in enumerate([(lev_low, C_GREEN), (lev_mid, C_ORANGE), (lev_high, C_RED)]):
        y = 8.5 - i * 3
        margin = entry / lev
        liq = entry * (1 - 1 / lev + 0.005)
        survived = liq < target
        result = 'SURVIVED' if survived else 'LIQUIDATED!'
        pnl = target - entry if survived else -margin
        rcol = C_GREEN if survived else C_RED

        ax.text(0.5, y + 0.8, f'{lev}x leverage', fontsize=14, fontweight='bold', color=col)
        ax.text(0.5, y, f'Margin: ${margin:,.0f}  |  Liq: ${liq:,.0f}', fontsize=10)
        ax.text(0.5, y - 0.8, f'{result}  |  P&L: ${pnl:+,.0f}', fontsize=12, fontweight='bold', color=rcol)
        if not survived:
            ax.text(8.5, y - 0.8, 'Wiped out', fontsize=11, ha='center', color=C_RED,
                    bbox=dict(boxstyle='round', facecolor='#ffe0e0', edgecolor=C_RED))

    plt.tight_layout()
    plt.show()

    print(f'\n  BTC ${entry:,} -> ${target:,.0f} ({drop_pct}% drop):')
    for lev in [lev_low, lev_mid, lev_high]:
        margin = entry / lev
        liq = entry * (1 - 1 / lev + 0.005)
        if liq >= target:
            print(f'    {lev}x: LIQUIDATED. Capital ${margin:,.0f} -> $0 (liq triggers at -${entry-liq:.0f})')
        else:
            print(f'    {lev}x: survived but lost ${entry-target:,.0f} (remaining ${margin-(entry-target):,.0f})')

interact(crash_simulator,
         entry=IntSlider(value=60000, min=20000, max=100000, step=5000, description='Entry($):'),
         drop_pct=IntSlider(value=50, min=5, max=90, step=5, description='Drop(%):'),
         lev_low=IntSlider(value=5, min=2, max=20, step=1, description='Low lev:'),
         lev_mid=IntSlider(value=20, min=10, max=50, step=5, description='Mid lev:'),
         lev_high=IntSlider(value=100, min=50, max=125, step=5, description='High lev:'));

## 13.3 血的教训 / 13.3 Hard-Earned Lessons

> **5x杠杆 / 5× leverage**: BTC跌20%才爆仓 → 扛得住大多数波动
> BTC must drop 20 % before liquidation → you survive most ordinary swings.
>
> **20x杠杆 / 20× leverage**: BTC跌5%就爆仓 → 一个正常回调就没了
> A 5 % drop wipes you out → a normal pullback is fatal.
>
> **100x杠杆 / 100× leverage**: BTC跌1%就爆仓 → 几分钟内可能归零
> A 1 % drop wipes you out → you can be zeroed inside minutes.

### 安全杠杆参考 / Safe-Leverage Cheat Sheet

| 品种 / Asset | 建议最高杠杆 / Suggested Max Leverage | 理由 / Reason |
|------|-------------|------|
| BTC | 3-5x / 3–5× | 日波动 3-5% 很常见 / 3–5 % daily swings are routine |
| ETH | 2-3x / 2–3× | 波动比BTC更大 / More volatile than BTC |
| 山寨币 / Altcoins | 1-2x / 1–2× | 动辄暴跌 20-50% / 20–50 % crashes are common |

> **下一章：综合模拟交易台** —— 把所有知识整合到一个面板
>
> **Next chapter: the all-in-one trading desk** — every tool we have learned, stitched into a single dashboard.